# Notebook 03 — Independent Intra-Plane Transformer Encoders

**Paper stage:** Section 2.3 — independent intra-plane Transformer encoders.

**Purpose:** Load the cached MedDINOv3 slice features and the *untrained RICA initialization* from Notebook 02, apply RICA to each plane, then pass each plane through its own Transformer encoder.

This notebook is an **architecture and implementation validation stage**, not a training or classification notebook. It does not claim that RICA or the Transformers have learned useful representations yet.

**Expected input per plane:** `(B, 64, 768)`  
**Base Transformer configuration from the paper:** 12 layers, 12 heads, embedding dimension 768, one CLS token, and 65 positional embeddings.

> Research-fidelity note: the paper does not fully specify every RICA implementation detail or every Transformer implementation detail (for example, some feed-forward/dropout settings). This notebook records explicit reconstruction choices rather than presenting unspecified values as paper facts.


## 1. Environment and reproducibility

In [1]:
import json
import random
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130
Device: cuda
GPU: Tesla T4


## 2. Locate Notebook 01 features and Notebook 02 RICA artifacts

Update `BASE_DIR` and `RICA_ARTIFACT_DIR` if your Kaggle input paths differ.

The feature files should contain the keys `axial_features`, `coronal_features`, `sagittal_features`, and `labels`. The RICA artifact directory should contain `rica_initialization.pt` and `rica_config.json`.

**Important:** In Kaggle, `/kaggle/working` from Notebook 02 is not automatically shared with Notebook 03. Save a Notebook 02 version with its output, then attach that output as an input here.


In [2]:
# EDIT THESE TWO PATHS to match the mounted Kaggle inputs.
BASE_DIR = Path(
    "/kaggle/input/notebooks/kamifiza/axial-centric-cross-plane-attention-for-3d-medical"
)
FEATURE_DIR = BASE_DIR / "features"

# Example only: replace this with the mounted output path from Notebook 02.
RICA_ARTIFACT_DIR = Path(
    "/kaggle/input/notebooks/kamifiza/02-plane-specific-representation-learning/rica_artifacts"
)

FEATURE_FILES = {
    "train": FEATURE_DIR / "features_train.pt",
    "val": FEATURE_DIR / "features_val.pt",
    "test": FEATURE_DIR / "features_test.pt",
}
RICA_CHECKPOINT_PATH = RICA_ARTIFACT_DIR / "rica_initialization.pt"
RICA_CONFIG_PATH = RICA_ARTIFACT_DIR / "rica_config.json"

for p in list(FEATURE_FILES.values()) + [RICA_CHECKPOINT_PATH, RICA_CONFIG_PATH]:
    print(("FOUND " if p.exists() else "MISSING"), p)

assert all(p.exists() for p in FEATURE_FILES.values()), (
    "Feature files not found. Correct BASE_DIR / FEATURE_DIR."
)
assert RICA_CHECKPOINT_PATH.exists(), (
    "RICA checkpoint not found. Attach Notebook 02 output and correct RICA_ARTIFACT_DIR."
)
assert RICA_CONFIG_PATH.exists(), "RICA config not found."


FOUND  /kaggle/input/notebooks/kamifiza/axial-centric-cross-plane-attention-for-3d-medical/features/features_train.pt
FOUND  /kaggle/input/notebooks/kamifiza/axial-centric-cross-plane-attention-for-3d-medical/features/features_val.pt
FOUND  /kaggle/input/notebooks/kamifiza/axial-centric-cross-plane-attention-for-3d-medical/features/features_test.pt
FOUND  /kaggle/input/notebooks/kamifiza/02-plane-specific-representation-learning/rica_artifacts/rica_initialization.pt
FOUND  /kaggle/input/notebooks/kamifiza/02-plane-specific-representation-learning/rica_artifacts/rica_config.json


## 3. Load and validate cached feature tensors

In [3]:
def load_split(path):
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(path, map_location="cpu")
    if not isinstance(obj, dict):
        raise TypeError(f"Expected a dictionary in {path}, got {type(obj)}")
    return obj

splits = {name: load_split(path) for name, path in FEATURE_FILES.items()}

PLANE_KEYS = {
    "axial": "axial_features",
    "coronal": "coronal_features",
    "sagittal": "sagittal_features",
}
EXPECTED_COUNTS = {"train": 971, "val": 161, "test": 610}
D, E = 64, 768

for split_name, data in splits.items():
    print(f"\\n{split_name}:")
    for plane, key in PLANE_KEYS.items():
        x = data[key]
        print(f"  {plane}: shape={tuple(x.shape)}, dtype={x.dtype}")
        assert tuple(x.shape) == (EXPECTED_COUNTS[split_name], D, E)
        assert torch.isfinite(x).all(), f"Non-finite values in {split_name}/{plane}"
    labels = data["labels"]
    assert len(labels) == EXPECTED_COUNTS[split_name]

print("\\nFeature validation passed.")


\ntrain:
  axial: shape=(971, 64, 768), dtype=torch.float32
  coronal: shape=(971, 64, 768), dtype=torch.float32
  sagittal: shape=(971, 64, 768), dtype=torch.float32
\nval:
  axial: shape=(161, 64, 768), dtype=torch.float32
  coronal: shape=(161, 64, 768), dtype=torch.float32
  sagittal: shape=(161, 64, 768), dtype=torch.float32
\ntest:
  axial: shape=(610, 64, 768), dtype=torch.float32
  coronal: shape=(610, 64, 768), dtype=torch.float32
  sagittal: shape=(610, 64, 768), dtype=torch.float32
\nFeature validation passed.


In [4]:
import json
import torch
from pathlib import Path

print("Config file exists:", RICA_CONFIG_PATH.exists())
print("Checkpoint exists:", RICA_CHECKPOINT_PATH.exists())

with open(RICA_CONFIG_PATH, "r") as f:
    rica_config_from_file = json.load(f)

rica_checkpoint = torch.load(
    RICA_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nRICA CONFIG:")
print(json.dumps(rica_config_from_file, indent=4))

print("\nCHECKPOINT KEYS:")
print(list(rica_checkpoint.keys()))

print("\nSTATE DICT EXAMPLE KEYS:")
print(list(rica_checkpoint["state_dict"].keys())[:10])

Config file exists: True
Checkpoint exists: True

RICA CONFIG:
{
    "pseudo_2d_layout": "(B,D,E) -> (B,1,D,E)",
    "in_channels": 1,
    "mid_channels": 1,
    "out_channels": 1,
    "kernel_size": 3,
    "ca_reduction": 32,
    "blocks_per_plane": 1,
    "plane_weights_shared": false,
    "status": "reconstruction; not verified original author code",
    "training_status": "initialized, not trained in this notebook"
}

CHECKPOINT KEYS:
['state_dict', 'config', 'seed', 'feature_metadata', 'training_status']

STATE DICT EXAMPLE KEYS:
['rica_axial.block.conv1.weight', 'rica_axial.block.bn1.weight', 'rica_axial.block.bn1.bias', 'rica_axial.block.bn1.running_mean', 'rica_axial.block.bn1.running_var', 'rica_axial.block.bn1.num_batches_tracked', 'rica_axial.block.conv2.weight', 'rica_axial.block.bn2.weight', 'rica_axial.block.bn2.bias', 'rica_axial.block.bn2.running_mean']


In [5]:
print("RICA CONFIG:")
print(rica_checkpoint["config"])

print("\nALL CHECKPOINT PARAMETERS:")
for key, value in rica_checkpoint["state_dict"].items():
    print(f"{key}: {tuple(value.shape)}")

RICA CONFIG:
{'pseudo_2d_layout': '(B,D,E) -> (B,1,D,E)', 'in_channels': 1, 'mid_channels': 1, 'out_channels': 1, 'kernel_size': 3, 'ca_reduction': 32, 'blocks_per_plane': 1, 'plane_weights_shared': False, 'status': 'reconstruction; not verified original author code', 'training_status': 'initialized, not trained in this notebook'}

ALL CHECKPOINT PARAMETERS:
rica_axial.block.conv1.weight: (1, 1, 3, 3)
rica_axial.block.bn1.weight: (1,)
rica_axial.block.bn1.bias: (1,)
rica_axial.block.bn1.running_mean: (1,)
rica_axial.block.bn1.running_var: (1,)
rica_axial.block.bn1.num_batches_tracked: ()
rica_axial.block.conv2.weight: (1, 1, 3, 3)
rica_axial.block.bn2.weight: (1,)
rica_axial.block.bn2.bias: (1,)
rica_axial.block.bn2.running_mean: (1,)
rica_axial.block.bn2.running_var: (1,)
rica_axial.block.bn2.num_batches_tracked: ()
rica_axial.block.ca.conv1.weight: (8, 1, 1, 1)
rica_axial.block.ca.conv1.bias: (8,)
rica_axial.block.ca.bn1.weight: (8,)
rica_axial.block.ca.bn1.bias: (8,)
rica_axial.bloc

## 4. Reconstruct the RICA modules

The following implementation mirrors the architecture/configuration documented in Notebook 02: Coordinate Attention, a two-convolution residual branch, and a Coordinate-Attention + 1×1 projection shortcut.

**Checkpoint compatibility is checked strictly below.** If strict loading reports missing or unexpected keys, do not ignore the error or silently continue. It means the class definitions/configuration here do not exactly match Notebook 02. In that case, copy the exact RICA class definitions from Notebook 02 into this section.


In [6]:

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6


class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.sigmoid = h_sigmoid(inplace=inplace)

    def forward(self, x):
        return x * self.sigmoid(x)


class CoordinateAttention(nn.Module):
    def __init__(self, inp, oup, reduction=32):
        super().__init__()

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()

        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()

        return identity * a_h * a_w


class RICAResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels=1,
        mid_channels=1,
        out_channels=1,
        kernel_size=3,
        reduction=32
    ):
        super().__init__()

        padding = kernel_size // 2

        self.conv1 = nn.Conv2d(
            in_channels, mid_channels,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            mid_channels, out_channels,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.ca = CoordinateAttention(
            in_channels, in_channels, reduction
        )

        self.projection = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = self.relu(residual)

        residual = self.conv2(residual)
        residual = self.bn2(residual)

        shortcut = self.ca(x)
        shortcut = self.projection(shortcut)

        return self.relu(residual + shortcut)


class PlaneRICA(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.block = RICAResidualBlock(
            in_channels=config["in_channels"],
            mid_channels=config["mid_channels"],
            out_channels=config["out_channels"],
            kernel_size=config["kernel_size"],
            reduction=config["ca_reduction"]
        )

    def forward(self, x):
        # (B, D, E) -> (B, 1, D, E)
        x = x.unsqueeze(1)
        x = self.block(x)
        return x.squeeze(1)


class ThreePlaneRICA(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.rica_axial = PlaneRICA(config)
        self.rica_coronal = PlaneRICA(config)
        self.rica_sagittal = PlaneRICA(config)

    def forward(self, axial, coronal, sagittal):
        return {
            "axial": self.rica_axial(axial),
            "coronal": self.rica_coronal(coronal),
            "sagittal": self.rica_sagittal(sagittal)
        }


## 5. Load the exact RICA initialization from Notebook 02

In [7]:

rica_config = rica_checkpoint["config"]

rica_model = ThreePlaneRICA(rica_config)

load_result = rica_model.load_state_dict(
    rica_checkpoint["state_dict"],
    strict=True
)

rica_model = rica_model.to(DEVICE)

print("RICA checkpoint loaded successfully!")
print("Strict loading:", load_result)
print("Training status:", rica_checkpoint["training_status"])
print(
    "Total RICA parameters:",
    sum(p.numel() for p in rica_model.parameters())
)


RICA checkpoint loaded successfully!
Strict loading: <All keys matched successfully>
Training status: untrained initialization
Total RICA parameters: 225


In [8]:

rica_model.eval()

with torch.no_grad():
    test_inputs = {
        plane: splits["train"][key][:4].float().to(DEVICE)
        for plane, key in PLANE_KEYS.items()
    }

    test_outputs = rica_model(
        test_inputs["axial"],
        test_inputs["coronal"],
        test_inputs["sagittal"]
    )

for plane, output in test_outputs.items():
    print(plane, output.shape)

    assert output.shape == (4, 64, 768)
    assert torch.isfinite(output).all()

print("All three RICA forward tests passed!")


axial torch.Size([4, 64, 768])
coronal torch.Size([4, 64, 768])
sagittal torch.Size([4, 64, 768])
All three RICA forward tests passed!


## 6. Apply RICA to one batch and validate its output

In [9]:
rica_model.eval()

with torch.no_grad():
    sample_inputs = {
        plane: splits["train"][key][:4].float().to(DEVICE)
        for plane, key in PLANE_KEYS.items()
    }
    sample_rica_outputs = rica_model(
        sample_inputs["axial"],
        sample_inputs["coronal"],
        sample_inputs["sagittal"],
    )

for plane, output in sample_rica_outputs.items():
    print(plane, tuple(output.shape), output.dtype)
    assert tuple(output.shape) == (4, D, E)
    assert torch.isfinite(output).all()

print("RICA forward validation passed.")


axial (4, 64, 768) torch.float32
coronal (4, 64, 768) torch.float32
sagittal (4, 64, 768) torch.float32
RICA forward validation passed.


## 7. Implement one intra-plane Transformer encoder

For each plane, the encoder:
1. receives 64 slice tokens of dimension 768;
2. prepends one learnable CLS token;
3. adds a learnable positional embedding for 65 tokens;
4. processes the sequence with a Transformer encoder;
5. returns the full token sequence and the CLS representation.

The paper specifies Base `N=12`, `H=12`, `E=768`. The paper does not fully state every implementation setting in the description available here. `FFN_DIM=2048` and `DROPOUT=0.1` below are explicit standard reconstruction choices; change them only if you verify different values in the paper/supplement.


In [10]:
EMBED_DIM = 768
NUM_LAYERS = 12
NUM_HEADS = 12
FFN_DIM = 2048       # reconstruction choice if not specified by the paper
DROPOUT = 0.1        # reconstruction choice if not specified by the paper
NUM_TOKENS = D + 1   # 64 slice tokens + 1 CLS token


class IntraPlaneTransformer(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_layers=NUM_LAYERS,
                 num_heads=NUM_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT,
                 num_tokens=NUM_TOKENS):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(embed_dim),
        )

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # x: (B, 64, 768)
        batch_size = x.shape[0]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, 65, 768)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        tokens = self.encoder(x)
        cls_representation = tokens[:, 0, :]
        return tokens, cls_representation


## 8. Create three independent Transformer branches

In [11]:
class ThreePlaneIntraTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer_axial = IntraPlaneTransformer()
        self.transformer_coronal = IntraPlaneTransformer()
        self.transformer_sagittal = IntraPlaneTransformer()

    def forward(self, rica_features):
        outputs = {}
        for plane, module_name in [
            ("axial", "transformer_axial"),
            ("coronal", "transformer_coronal"),
            ("sagittal", "transformer_sagittal"),
        ]:
            tokens, cls_rep = getattr(self, module_name)(rica_features[plane])
            outputs[plane] = {
                "tokens": tokens,
                "cls": cls_rep,
            }
        return outputs


intra_transformers = ThreePlaneIntraTransformer().to(DEVICE)

# Verify that the three branches do not share parameter objects.
parameter_ids = {
    plane: {id(p) for p in getattr(intra_transformers, module_name).parameters()}
    for plane, module_name in [
        ("axial", "transformer_axial"),
        ("coronal", "transformer_coronal"),
        ("sagittal", "transformer_sagittal"),
    ]
}
assert parameter_ids["axial"].isdisjoint(parameter_ids["coronal"])
assert parameter_ids["axial"].isdisjoint(parameter_ids["sagittal"])
assert parameter_ids["coronal"].isdisjoint(parameter_ids["sagittal"])

print("Three independent Transformer branches created.")
print("Parameters per branch:",
      sum(p.numel() for p in intra_transformers.transformer_axial.parameters()))


Three independent Transformer branches created.
Parameters per branch: 66220032


## 9. End-to-end forward test: features → RICA → three Transformers

In [12]:
rica_model.eval()
intra_transformers.eval()

with torch.no_grad():
    x = {
        plane: splits["train"][key][:2].float().to(DEVICE)
        for plane, key in PLANE_KEYS.items()
    }
    rica_features = rica_model(x["axial"], x["coronal"], x["sagittal"])
    transformer_outputs = intra_transformers(rica_features)

for plane in ["axial", "coronal", "sagittal"]:
    tokens = transformer_outputs[plane]["tokens"]
    cls = transformer_outputs[plane]["cls"]
    print(f"{plane}: tokens={tuple(tokens.shape)}, CLS={tuple(cls.shape)}")
    assert tuple(tokens.shape) == (2, 65, 768)
    assert tuple(cls.shape) == (2, 768)
    assert torch.isfinite(tokens).all()
    assert torch.isfinite(cls).all()

print("End-to-end forward test passed.")


axial: tokens=(2, 65, 768), CLS=(2, 768)
coronal: tokens=(2, 65, 768), CLS=(2, 768)
sagittal: tokens=(2, 65, 768), CLS=(2, 768)
End-to-end forward test passed.


## 10. Gradient-flow test through RICA and all three Transformer branches

In [13]:
# Use copies so this test does not alter the models that will be saved.
rica_grad_model = copy.deepcopy(rica_model).to(DEVICE).train()
transformer_grad_model = copy.deepcopy(intra_transformers).to(DEVICE).train()

x_grad = {
    plane: splits["train"][key][:2].float().to(DEVICE)
    for plane, key in PLANE_KEYS.items()
}

rica_grad_model.zero_grad(set_to_none=True)
transformer_grad_model.zero_grad(set_to_none=True)

rica_out = rica_grad_model(
    x_grad["axial"], x_grad["coronal"], x_grad["sagittal"]
)
trans_out = transformer_grad_model(rica_out)

loss = sum(trans_out[p]["cls"].square().mean()
           for p in ["axial", "coronal", "sagittal"])
loss.backward()

for module_name in ["rica_axial", "rica_coronal", "rica_sagittal"]:
    module = getattr(rica_grad_model, module_name)
    grads = [p.grad for p in module.parameters() if p.requires_grad]
    assert any(g is not None for g in grads), f"No gradients in {module_name}"
    assert all(torch.isfinite(g).all() for g in grads if g is not None)
    print(module_name, "gradient test passed")

for module_name in [
    "transformer_axial", "transformer_coronal", "transformer_sagittal"
]:
    module = getattr(transformer_grad_model, module_name)
    grads = [p.grad for p in module.parameters() if p.requires_grad]
    assert any(g is not None for g in grads), f"No gradients in {module_name}"
    assert all(torch.isfinite(g).all() for g in grads if g is not None)
    print(module_name, "gradient test passed")

print("Gradient flow through all six modules passed.")


rica_axial gradient test passed
rica_coronal gradient test passed
rica_sagittal gradient test passed
transformer_axial gradient test passed
transformer_coronal gradient test passed
transformer_sagittal gradient test passed
Gradient flow through all six modules passed.


## 11. Save the initialized intra-plane Transformer stage

This saves:
- the loaded RICA state (still untrained);
- the three independently initialized Transformer branches;
- the architecture configuration and metadata.

The checkpoint is an **initialization checkpoint**, not a trained model.


In [14]:
OUTPUT_DIR = Path("/kaggle/working/intra_plane_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRANSFORMER_CONFIG = {
    "embed_dim": EMBED_DIM,
    "num_layers": NUM_LAYERS,
    "num_heads": NUM_HEADS,
    "ffn_dim": FFN_DIM,
    "dropout": DROPOUT,
    "num_input_slices": D,
    "num_tokens_with_cls": NUM_TOKENS,
    "activation": "relu",
    "batch_first": True,
    "norm_first": False,
    "ffn_dim_status": "reconstruction choice; verify against paper/supplement",
    "dropout_status": "reconstruction choice; verify against paper/supplement",
    "independent_plane_weights": True,
}

artifact = {
    "rica_state_dict": rica_model.state_dict(),
    "rica_config": rica_config,
    "transformer_state_dict": intra_transformers.state_dict(),
    "transformer_config": TRANSFORMER_CONFIG,
    "seed": SEED,
    "feature_metadata": json.load(open(FEATURE_DIR / "meta.json", "r")),
    "training_status": "untrained initialization; architecture validation only",
}

checkpoint_out = OUTPUT_DIR / "intra_plane_initialization.pt"
config_out = OUTPUT_DIR / "intra_plane_config.json"

torch.save(artifact, checkpoint_out)

with open(config_out, "w") as f:
    json.dump({
        "rica_config": rica_config,
        "transformer_config": TRANSFORMER_CONFIG,
        "training_status": artifact["training_status"],
    }, f, indent=2)

print("Saved:", checkpoint_out)
print("Saved:", config_out)

Saved: /kaggle/working/intra_plane_artifacts/intra_plane_initialization.pt
Saved: /kaggle/working/intra_plane_artifacts/intra_plane_config.json


## 12. Completion report and limitations

### Completed in this notebook
- Loaded and validated Notebook 01 cached features.
- Loaded the RICA initialization from Notebook 02.
- Applied the three independent RICA modules to real feature batches.
- Implemented three independent intra-plane Transformer encoders.
- Added CLS tokens and positional embeddings.
- Verified output shapes, finite values, independent parameters, and gradient flow.
- Saved the initialized stage.

### Not completed
- RICA has **not** been trained.
- The Transformer encoders have **not** been trained.
- No classification head, cross-plane attention, or accuracy evaluation is included.
- The feed-forward dimension and dropout are reconstruction choices unless verified from the paper/supplement.
- Cached features are unaugmented. The paper's volume-level TorchIO augmentations are not reproduced by simply augmenting cached feature tensors.

### Next architectural stage
Implement **axial-centric cross-plane attention** using the three intra-plane Transformer token sequences/representations, following Section 2.4 of the paper. The full architecture should later be trained jointly rather than treating these random initializations as learned representations.
